# Batch Processing for Multiple LIF Containers

This notebook processes `.lif` containers in batch mode with no visualization.

Outputs are written to `../results/<lif_container_id>/<lif_image_name>.csv`.

It reuses segmentation and feature-extraction utilities from `../src` and includes researcher-friendly progress/logging with `tqdm`.

In [ ]:
import json
import re
import sys
import time
import traceback
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.append("../src")

from utils.io import (
    calculate_rescale_factor,
    explore_lif_container,
    get_voxel_spacing_zyx_um,
    list_containers,
    load_lif_image,
)
from utils.inference import predict_tiled_unet
from utils.segmentation import (
    fill_root_3d,
    generate_rough_root_3d,
    predict_nuclei_labels,
    simulate_fluo_from_bf,
    smooth_outer_root_surface_3d,
    wrap_outer_root_surface,
)
from utils.feature_extraction import (
    calculate_distance_to_root_surface,
    classify_root_cap_nuclei,
    compute_fret_ratios,
    extract_nuclei_depth,
    extract_nuclei_features_per_marker,
    map_root_body_depth_clusters_to_tissue_layers,
    merge_root_cap_into_tissue_layers,
)

pd.set_option("display.max_columns", None)

In [ ]:
# Paths
RAW_DATA_DIRECTORY = Path("../raw_data")
RESULTS_ROOT = Path("../results")
MODEL_DIR = Path("../plantseg_models/lightsheet_3D_unet_root_ds3x")

# Marker configuration: (marker_name, channel_index, role)
MARKERS = (
    ("edCerulean_CTRL", 0, "DD"),
    ("edCitrine_FRET", 1, "DA"),
    ("edCitrine_CTRL", 2, "DD"),
    ("brightfield", 3, "root_structure"),
)

# Segmentation and depth settings
NUCLEI_CHANNEL = 2
MIN_MAX_NUCLEI_VOLUME = (250, 4000)
ROOT_PROBABILITY_THRESHOLD = 0.9
ROOT_OCCUPANCY_THRESHOLD = 0.9
ROOT_FILL_SLICE_AWARE = True
ROOT_SMOOTH_EROSION = 5
ROOT_SMOOTHING = 3
ROOT_WRAP_PERCENTAGE_THRESHOLD = 5.0
ROOT_WRAP_EDT_THRESHOLD = 15.0
DEPTH_PAD_FULL_ROOT = False

# Inference settings
INFERENCE_PATCH = (80, 160, 160)
INFERENCE_PATCH_HALO = (8, 16, 16)
INFERENCE_STRIDE_RATIO = 0.75
INFERENCE_BATCH_SIZE = 1
INFERENCE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INFERENCE_USE_AMP = True

# Batch behavior
CONTAINER_INDICES: list[int] | None = None   # e.g. [0, 2, 3]
IMAGE_INDICES: list[int] | None = None       # e.g. [0, 1, 5]
OVERWRITE_CSV = False

In [ ]:
def sanitize_filename(name: str) -> str:
    sanitized = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", str(name)).strip()
    return sanitized if sanitized else "unnamed"


def log_step(
    level: str,
    container_id: str,
    image_name: str,
    stage: str,
    msg: str,
    elapsed_s: float | None = None,
    details: dict[str, Any] | None = None,
) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    elapsed_text = "na" if elapsed_s is None else f"{elapsed_s:.2f}"
    details_text = "na" if details is None else json.dumps(details, default=str)

    line = (
        f"[{timestamp}] | {level} | container={container_id} | image={image_name} "
        f"| stage={stage} | msg={msg} | elapsed_s={elapsed_text} | details={details_text}"
    )
    tqdm.write(line)


def save_image_csv(df: pd.DataFrame, csv_path: Path) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)


def process_single_image(
    lif_path: str,
    image_index: int,
    lif_container_id: str,
    config: dict[str, Any],
) -> dict[str, Any]:
    image_start = time.perf_counter()
    image_name = f"image_{image_index}"

    try:
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "image_start",
            "starting image processing",
            elapsed_s=0.0,
        )

        stage_t0 = time.perf_counter()
        lif_image, lif_image_name, xml_metadata = load_lif_image(lif_path, image_index)
        image_name = lif_image_name
        safe_image_name = sanitize_filename(lif_image_name)
        csv_path = config["results_root"] / lif_container_id / f"{safe_image_name}.csv"

        elapsed = time.perf_counter() - image_start
        stage_elapsed = time.perf_counter() - stage_t0
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "load",
            "image and metadata loaded",
            elapsed_s=elapsed,
            details={"stage_s": round(stage_elapsed, 2), "image_index": image_index},
        )

        if csv_path.exists() and not config["overwrite_csv"]:
            elapsed = time.perf_counter() - image_start
            log_step(
                "WARN",
                lif_container_id,
                image_name,
                "skip",
                "csv exists and overwrite disabled",
                elapsed_s=elapsed,
                details={"csv": str(csv_path)},
            )
            return {
                "status": "skipped",
                "lif_image_name": lif_image_name,
                "csv_path": csv_path,
            }

        descriptor_dict = {
            "lif_container_id": lif_container_id,
            "lif_image_name": lif_image_name,
        }

        stage_t0 = time.perf_counter()
        rescale_factor = calculate_rescale_factor(xml_metadata)
        nuclei_labels = predict_nuclei_labels(
            lif_image,
            rescale_factor,
            config["nuclei_channel"],
            config["min_max_nuclei_volume"],
            visualize=False,
        )
        elapsed = time.perf_counter() - image_start
        stage_elapsed = time.perf_counter() - stage_t0
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "nuclei",
            "nuclei segmentation complete",
            elapsed_s=elapsed,
            details={"stage_s": round(stage_elapsed, 2), "n_labels": int(nuclei_labels.max())},
        )

        stage_t0 = time.perf_counter()
        props_df = extract_nuclei_features_per_marker(
            nuclei_labels,
            lif_image,
            config["markers"],
            descriptor_dict,
        )
        props_df = compute_fret_ratios(props_df, config["markers"])
        elapsed = time.perf_counter() - image_start
        stage_elapsed = time.perf_counter() - stage_t0
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "features",
            "feature extraction and FRET computation complete",
            elapsed_s=elapsed,
            details={"stage_s": round(stage_elapsed, 2), "n_rows": int(len(props_df))},
        )

        stage_t0 = time.perf_counter()
        sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, config["markers"])
        root_pmaps = predict_tiled_unet(
            raw=sim_fluo_cell_walls,
            input_layout="ZYX",
            model_dir=config["model_dir"],
            patch=config["inference_patch"],
            patch_halo=config["inference_patch_halo"],
            stride_ratio=config["inference_stride_ratio"],
            batch_size=config["inference_batch_size"],
            device=config["inference_device"],
            use_amp=config["inference_use_amp"],
        )
        rough_root_3d = generate_rough_root_3d(
            root_pmaps,
            nuclei_labels,
            probability_threshold=config["root_probability_threshold"],
            visualize=False,
            remove_nonconnected_components=False,
        )
        filled_root_3d = fill_root_3d(
            rough_root_3d,
            occupancy_threshold=config["root_occupancy_threshold"],
            slice_aware_filling=config["root_fill_slice_aware"],
            visualize=False,
        )
        smooth_root_3d = smooth_outer_root_surface_3d(
            filled_root_3d,
            erosion=config["root_smooth_erosion"],
            smoothing=config["root_smoothing"],
            visualize=False,
        )
        props_df = classify_root_cap_nuclei(props_df)
        root_body_3d_mask = wrap_outer_root_surface(
            nuclei_labels,
            smooth_root_3d,
            props_df,
            percentage_threshold=config["root_wrap_percentage_threshold"],
            edt_threshold=config["root_wrap_edt_threshold"],
            visualize=False,
        )
        elapsed = time.perf_counter() - image_start
        stage_elapsed = time.perf_counter() - stage_t0
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "root_mask",
            "root mask and root-part assignment complete",
            elapsed_s=elapsed,
            details={"stage_s": round(stage_elapsed, 2)},
        )

        stage_t0 = time.perf_counter()
        spacing_zyx_um = get_voxel_spacing_zyx_um(xml_metadata)
        nuclei_depth_map, is_flooded, flooded_planes = calculate_distance_to_root_surface(
            nuclei_labels,
            root_body_3d_mask,
            pad_full_root=config["depth_pad_full_root"],
            spacing_zyx_um=spacing_zyx_um,
            visualize=False,
        )
        depth_df = extract_nuclei_depth(nuclei_labels, nuclei_depth_map)
        props_df = props_df.merge(depth_df, on="label")

        depth_clusters_df = map_root_body_depth_clusters_to_tissue_layers(props_df)
        props_df = merge_root_cap_into_tissue_layers(props_df, depth_clusters_df)
        elapsed = time.perf_counter() - image_start
        stage_elapsed = time.perf_counter() - stage_t0
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "depth",
            "depth map and tissue layers computed",
            elapsed_s=elapsed,
            details={
                "stage_s": round(stage_elapsed, 2),
                "is_flooded": bool(is_flooded),
                "flooded_planes": int(len(flooded_planes)),
            },
        )

        stage_t0 = time.perf_counter()
        save_image_csv(props_df, csv_path)
        elapsed = time.perf_counter() - image_start
        stage_elapsed = time.perf_counter() - stage_t0
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "write_csv",
            "csv written",
            elapsed_s=elapsed,
            details={"stage_s": round(stage_elapsed, 2), "csv": str(csv_path)},
        )

        total_elapsed = time.perf_counter() - image_start
        log_step(
            "INFO",
            lif_container_id,
            image_name,
            "done",
            "image processing completed",
            elapsed_s=total_elapsed,
            details={"n_rows": int(len(props_df))},
        )

        return {
            "status": "success",
            "lif_image_name": lif_image_name,
            "csv_path": csv_path,
            "n_rows": int(len(props_df)),
        }

    except Exception as exc:
        total_elapsed = time.perf_counter() - image_start
        log_step(
            "ERROR",
            lif_container_id,
            image_name,
            "failed",
            "processing error",
            elapsed_s=total_elapsed,
            details={"error_type": type(exc).__name__, "error": str(exc)},
        )
        tqdm.write(traceback.format_exc())
        return {
            "status": "failed",
            "lif_image_name": image_name,
            "error_type": type(exc).__name__,
            "error": str(exc),
        }


def run_batch(config: dict[str, Any]) -> dict[str, Any]:
    batch_start = time.perf_counter()
    config["results_root"].mkdir(parents=True, exist_ok=True)

    lif_containers = list_containers(str(config["raw_data_directory"]), file_format="lif")
    if not lif_containers:
        raise FileNotFoundError(
            f"No .lif containers found in '{config['raw_data_directory']}'."
        )

    if config["container_indices"] is None:
        selected_containers = lif_containers
    else:
        selected_containers = [lif_containers[i] for i in config["container_indices"]]

    processed = 0
    success = 0
    skipped = 0
    failed = 0
    failures: list[dict[str, str]] = []

    container_pbar = tqdm(selected_containers, desc="Containers", unit="container")
    for lif_path in container_pbar:
        nr_imgs, lif_container_id = explore_lif_container(lif_path, display=False)

        if config["image_indices"] is None:
            image_indices = list(range(nr_imgs))
        else:
            image_indices = [i for i in config["image_indices"] if i < nr_imgs]

        image_pbar = tqdm(
            image_indices,
            desc=f"Images[{lif_container_id}]",
            unit="image",
            leave=False,
        )

        for image_index in image_pbar:
            result = process_single_image(
                lif_path=lif_path,
                image_index=image_index,
                lif_container_id=lif_container_id,
                config=config,
            )

            processed += 1
            status = result["status"]
            if status == "success":
                success += 1
            elif status == "skipped":
                skipped += 1
            else:
                failed += 1
                failures.append(
                    {
                        "container": lif_container_id,
                        "image": str(result.get("lif_image_name", f"image_{image_index}")),
                        "error_type": str(result.get("error_type", "UnknownError")),
                        "error": str(result.get("error", "Unknown error")),
                    }
                )

            image_pbar.set_postfix(
                processed=processed,
                success=success,
                skipped=skipped,
                failed=failed,
            )
            container_pbar.set_postfix(
                container=lif_container_id,
                processed=processed,
                success=success,
                skipped=skipped,
                failed=failed,
            )

    elapsed_min = (time.perf_counter() - batch_start) / 60.0
    tqdm.write(
        "BATCH_SUMMARY"
        f" | containers={len(selected_containers)}"
        f" | images_total={processed}"
        f" | processed={processed}"
        f" | success={success}"
        f" | skipped={skipped}"
        f" | failed={failed}"
        f" | elapsed_min={elapsed_min:.2f}"
    )

    if failures:
        for item in failures:
            tqdm.write(
                "FAILED_ITEM"
                f" | container={item['container']}"
                f" | image={item['image']}"
                f" | error_type={item['error_type']}"
                f" | error={item['error']}"
            )

    return {
        "containers": len(selected_containers),
        "images_total": processed,
        "processed": processed,
        "success": success,
        "skipped": skipped,
        "failed": failed,
        "elapsed_min": elapsed_min,
        "failures": failures,
    }


CONFIG = {
    "raw_data_directory": RAW_DATA_DIRECTORY,
    "results_root": RESULTS_ROOT,
    "model_dir": MODEL_DIR,
    "markers": MARKERS,
    "nuclei_channel": NUCLEI_CHANNEL,
    "min_max_nuclei_volume": MIN_MAX_NUCLEI_VOLUME,
    "root_probability_threshold": ROOT_PROBABILITY_THRESHOLD,
    "root_occupancy_threshold": ROOT_OCCUPANCY_THRESHOLD,
    "root_fill_slice_aware": ROOT_FILL_SLICE_AWARE,
    "root_smooth_erosion": ROOT_SMOOTH_EROSION,
    "root_smoothing": ROOT_SMOOTHING,
    "root_wrap_percentage_threshold": ROOT_WRAP_PERCENTAGE_THRESHOLD,
    "root_wrap_edt_threshold": ROOT_WRAP_EDT_THRESHOLD,
    "depth_pad_full_root": DEPTH_PAD_FULL_ROOT,
    "inference_patch": INFERENCE_PATCH,
    "inference_patch_halo": INFERENCE_PATCH_HALO,
    "inference_stride_ratio": INFERENCE_STRIDE_RATIO,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
    "inference_device": INFERENCE_DEVICE,
    "inference_use_amp": INFERENCE_USE_AMP,
    "container_indices": CONTAINER_INDICES,
    "image_indices": IMAGE_INDICES,
    "overwrite_csv": OVERWRITE_CSV,
}

In [ ]:
batch_results = run_batch(CONFIG)
batch_results